1. This cell loads all necessary components and sets the configuration variables. LANGUAGE controls which language documents are loaded, LABEL_LANG controls which language the label descriptors are fetched in, and LABEL_CONDITION is used for labelling the saved results. Setting LABEL_LANG equal to LANGUAGE runs the native label condition, while keeping LABEL_LANG as 'en' runs the English label baseline. The cell also detects the available compute device, prioritising Apple MPS GPU acceleration on Apple Silicon hardware.

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import requests
from tqdm import tqdm
import pandas as pd
import torch
import json
import os

# ── Configuration ─────────────────────────────────────────
LANGUAGE = 'en'        # change to 'fr', 'nl', 'de'
LANGUAGE_NAME = 'English'
LABEL_LANG = 'en'      # change to match LANGUAGE for native label condition
LABEL_CONDITION = 'EN labels'  # change to 'native labels' for native condition
CANDIDATE_K = 100      # number of candidates retrieved before reranking
# ──────────────────────────────────────────────────────────

# Device detection for Apple Silicon
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS GPU")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU — this will be slow")

dataset = load_dataset('coastalcph/multi_eurlex', language, split='test',
                       label_level='all_levels', trust_remote_code=True)

classlabel = dataset.features["labels"].feature
print(f"Number of labels: {len(classlabel.names)}")

url = "https://raw.githubusercontent.com/nlpaueb/multi-eurlex/master/data/eurovoc_descriptors.json"
eurovoc_concepts = requests.get(url).json()
label_ids = classlabel.names

label_descriptors_raw = [eurovoc_concepts[label_id][LABEL_LANG] for label_id in label_ids]
label_descriptors = ["passage: " + d for d in label_descriptors_raw]

embedding_model = SentenceTransformer('intfloat/multilingual-e5-small')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'coastalcph/multi_eurlex' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Using Apple MPS GPU


Using the latest cached version of the dataset since coastalcph/multi_eurlex couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'en-label_level=all_levels' at /Users/vincent/.cache/huggingface/datasets/coastalcph___multi_eurlex/en-label_level=all_levels/1.0.0/5a12a7463045d4dcb12896b478c09b5a8a131a02b7e7bce059ba7ececc6584ee (last modified on Mon Feb  2 17:46:09 2026).


Number of labels: 7390


2. This cell encodes all 7,390 label descriptors into embedding vectors using the multilingual-E5-small model, with the passage prefix applied as required by the E5 model. Label embeddings are computed once and reused for all documents.

In [2]:
label_embeddings = embedding_model.encode(
    label_descriptors,
    show_progress_bar=True,
    batch_size=32
)
print(f"Label embeddings shape: {label_embeddings.shape}")

Batches:   0%|          | 0/231 [00:00<?, ?it/s]

Label embeddings shape: (7390, 384)


3. This cell encodes all 5,000 documents in batches using the query prefix required by E5. Batch encoding is more efficient than encoding documents one at a time.

In [3]:
texts = ["query: " + doc['text'] for doc in dataset]
doc_embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)
print(f"Document embeddings shape: {doc_embeddings.shape}")

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Document embeddings shape: (5000, 384)


4. This cell loads the BGE-M3 reranker model. The M3 reranker is a cross-encoder that supports 100+ languages including French, Dutch and German, making it suitable for the multilingual evaluation in this thesis. Setting use_fp16=True reduces memory usage and speeds up computation with negligible impact on accuracy. Install FlagEmbedding first if needed with pip install FlagEmbedding.

In [4]:
from FlagEmbedding import FlagReranker

reranker = FlagReranker('BAAI/bge-m3', use_fp16=True)
print("Reranker loaded ✓")

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at BAAI/bge-m3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Reranker loaded ✓


5. This cell defines the three evaluation metrics used throughout the thesis — Precision@k, Recall@k, and NDCG@k. These functions are model-agnostic and identical across all evaluation notebooks.

In [5]:
def precision_at_k(y_true, y_pred, k):
    """Precision@k: fraction of top-k predictions that are correct"""
    top_k = y_pred[:k]
    relevant = sum(1 for label in top_k if label in y_true)
    return relevant / k

def recall_at_k(y_true, y_pred, k):
    """Recall@k: fraction of true labels found in top-k"""
    if len(y_true) == 0:
        return 0.0
    top_k = y_pred[:k]
    relevant = sum(1 for label in top_k if label in y_true)
    return relevant / len(y_true)

def ndcg_at_k(y_true, y_pred, k):
    """NDCG@k: Normalized Discounted Cumulative Gain"""
    top_k = y_pred[:k]
    dcg = sum((1 if label in y_true else 0) / np.log2(i + 2)
              for i, label in enumerate(top_k))
    ideal_k = min(len(y_true), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_k))
    return dcg / idcg if idcg > 0 else 0.0

print("Metric functions defined ✓")

Metric functions defined ✓


6. This cell implements the two-stage retrieval and reranking pipeline. In the first stage, cosine similarity between document and label embeddings is used to retrieve the top 100 candidate labels efficiently. In the second stage, the BGE-M3 cross-encoder reranker scores each document-candidate pair directly, reading both texts together to produce a more accurate relevance score. The final ranking is determined by the reranker scores. This two-stage design balances efficiency — embedding-based retrieval is fast over all 7,390 labels — with accuracy, since the cross-encoder can only be applied to a smaller candidate set due to its higher computational cost.

In [6]:
k_values = [5, 10, 20, 50, 100]
results = {
    'precision': {k: [] for k in k_values},
    'recall': {k: [] for k in k_values},
    'ndcg': {k: [] for k in k_values}
}

for idx, doc in enumerate(tqdm(dataset, desc="Evaluating")):
    true_labels = set(doc['labels'])
    if len(true_labels) == 0:
        continue

    # Stage 1: retrieve top CANDIDATE_K labels by cosine similarity
    similarities = cosine_similarity([doc_embeddings[idx]], label_embeddings)[0]
    top_candidate_indices = np.argsort(similarities)[::-1][:CANDIDATE_K]

    # Stage 2: rerank candidates using cross-encoder
    doc_text = doc['text']
    pairs = [[doc_text, label_descriptors_raw[i]] for i in top_candidate_indices]
    rerank_scores = reranker.compute_score(pairs, batch_size=32)

    # Sort candidates by reranker score
    reranked_order = np.argsort(rerank_scores)[::-1]
    reranked_predictions = top_candidate_indices[reranked_order]

    for k in k_values:
        results['precision'][k].append(precision_at_k(true_labels, reranked_predictions, k))
        results['recall'][k].append(recall_at_k(true_labels, reranked_predictions, k))
        results['ndcg'][k].append(ndcg_at_k(true_labels, reranked_predictions, k))

Evaluating:   0%|          | 0/5000 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 63.24it/s]





pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 82.94it/s]





Evaluating:   0%|          | 2/5000 [00:33<23:00:55, 16.58s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 28.20it/s]





Evaluating:   0%|          | 3/5000 [00:50<23:12:40, 16.72s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 23.55it/s]





Evaluating:   1%|          | 29/5000 [14:02<40:08:16, 29.07s/it]


KeyboardInterrupt: 

7. Displays Results

In [ ]:
summary_data = []
for k in k_values:
    summary_data.append({
        'k': k,
        'Precision': f"{np.mean(results['precision'][k]):.4f} ± {np.std(results['precision'][k]):.4f}",
        'Recall': f"{np.mean(results['recall'][k]):.4f} ± {np.std(results['recall'][k]):.4f}",
        'NDCG': f"{np.mean(results['ndcg'][k]):.4f} ± {np.std(results['ndcg'][k]):.4f}"
    })
summary_df = pd.DataFrame(summary_data)
print(f"\nResults for E5 + BGE-M3 reranker | {LANGUAGE_NAME} docs | {LABEL_CONDITION}")
print(summary_df.to_string(index=False))

8. This cell saves the complete evaluation results to a JSON file. The filename is generated automatically from the language and label condition variables, producing files like results_e5_reranked_fr_native_labels.json. The candidate_k field records how many candidates were retrieved before reranking, which is important context for interpreting the results.

In [ ]:
results_to_save = {
    'model': 'multilingual-e5-small + bge-m3-reranker',
    'dataset': 'MultiEURLEX',
    'language': f'{LANGUAGE_NAME} ({LABEL_CONDITION})',
    'candidate_k': CANDIDATE_K,
    'num_documents': len(dataset),
    'num_labels': len(label_descriptors),
    'metrics': {
        metric_name: {
            k: {
                'mean': float(np.mean(scores)),
                'std': float(np.std(scores)),
                'values': [float(x) for x in scores]
            }
            for k, scores in metric_data.items()
        }
        for metric_name, metric_data in results.items()
    }
}

filename = f'results_e5_reranked_{LANGUAGE}_{LABEL_CONDITION.replace(" ", "_").lower()}.json'
with open(filename, 'w') as f:
    json.dump(results_to_save, f, indent=2)
print(f"Saved to {filename}")